# День 1 — Токенизация

Цель: понять, как текст превращается в токены, input_ids и attention_mask.

Модель: distilbert-base-uncased

In [1]:
from transformers import AutoTokenizer
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Model:", model_name)
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)

C:\ProgramData\anaconda3\envs\transformers_overall\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: distilbert-base-uncased
Vocab size: 30522
Model max length: 512


In [2]:
text = "This movie was absolutely amazing!"

tokens = tokenizer(text)

print(tokens)

{'input_ids': [101, 2023, 3185, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [3]:
input_ids = tokens["input_ids"]

print(f"Количество токенов: {len(input_ids)}")
print(f"Input IDs: {input_ids}")

decoded = tokenizer.decode(input_ids)
print(f"Декодировано: {decoded}")

Количество токенов: 8
Input IDs: [101, 2023, 3185, 2001, 7078, 6429, 999, 102]
Декодировано: [CLS] this movie was absolutely amazing! [SEP]


In [4]:
def tokenize_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

texts = [
    "This movie was great!",
    "Terrible movie, waste of time."
]

batch_tokens = tokenize_texts(texts)

print("Input IDs shape:", batch_tokens["input_ids"].shape)
print("Input IDs:")
print(batch_tokens["input_ids"])

print("\nAttention mask:")
print(batch_tokens["attention_mask"])

Input IDs shape: torch.Size([2, 9])
Input IDs:
tensor([[ 101, 2023, 3185, 2001, 2307,  999,  102,    0,    0],
        [ 101, 6659, 3185, 1010, 5949, 1997, 2051, 1012,  102]])

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [5]:
print(f"CLS token: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})")
print(f"SEP token: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

CLS token: [CLS] (ID: 101)
SEP token: [SEP] (ID: 102)
PAD token: [PAD] (ID: 0)


In [6]:
def explain_tokenization(text, tokenizer):
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    print(f"Исходный текст: {text}")
    print("-" * 60)

    for token, token_id in zip(tokens, ids):
        print(f"{token:20} -> {token_id}")

    print("-" * 60)
    print(f"Количество токенов без специальных токенов: {len(tokens)}")

    encoded = tokenizer(text)
    print(f"Количество токенов со специальными токенами: {len(encoded['input_ids'])}")
    print(f"Decoded: {tokenizer.decode(encoded['input_ids'])}")

In [7]:
explain_tokenization("Transformers are amazing!", tokenizer)

Исходный текст: Transformers are amazing!
------------------------------------------------------------
transformers         -> 19081
are                  -> 2024
amazing              -> 6429
!                    -> 999
------------------------------------------------------------
Количество токенов без специальных токенов: 4
Количество токенов со специальными токенами: 6
Decoded: [CLS] transformers are amazing! [SEP]


In [8]:
explain_tokenization("This movie was amazing!", tokenizer)
explain_tokenization("THIS MOVIE WAS AMAZING!", tokenizer)

Исходный текст: This movie was amazing!
------------------------------------------------------------
this                 -> 2023
movie                -> 3185
was                  -> 2001
amazing              -> 6429
!                    -> 999
------------------------------------------------------------
Количество токенов без специальных токенов: 5
Количество токенов со специальными токенами: 7
Decoded: [CLS] this movie was amazing! [SEP]
Исходный текст: THIS MOVIE WAS AMAZING!
------------------------------------------------------------
this                 -> 2023
movie                -> 3185
was                  -> 2001
amazing              -> 6429
!                    -> 999
------------------------------------------------------------
Количество токенов без специальных токенов: 5
Количество токенов со специальными токенами: 7
Decoded: [CLS] this movie was amazing! [SEP]


In [9]:
explain_tokenization("This movie was unbelievably good!", tokenizer)

Исходный текст: This movie was unbelievably good!
------------------------------------------------------------
this                 -> 2023
movie                -> 3185
was                  -> 2001
un                   -> 4895
##bel                -> 8671
##ie                 -> 2666
##va                 -> 3567
##bly                -> 6321
good                 -> 2204
!                    -> 999
------------------------------------------------------------
Количество токенов без специальных токенов: 10
Количество токенов со специальными токенами: 12
Decoded: [CLS] this movie was unbelievably good! [SEP]


In [10]:
explain_tokenization("Этот фильм был отличным!", tokenizer)

Исходный текст: Этот фильм был отличным!
------------------------------------------------------------
э                    -> 1208
##т                  -> 22919
##о                  -> 14150
##т                  -> 22919
ф                    -> 1199
##и                  -> 10325
##л                  -> 29436
##ь                  -> 23742
##м                  -> 29745
б                    -> 1181
##ы                  -> 29113
##л                  -> 29436
о                    -> 1193
##т                  -> 22919
##л                  -> 29436
##и                  -> 10325
##ч                  -> 29752
##н                  -> 18947
##ы                  -> 29113
##м                  -> 29745
!                    -> 999
------------------------------------------------------------
Количество токенов без специальных токенов: 21
Количество токенов со специальными токенами: 23
Decoded: [CLS] этот фильм был отличным! [SEP]
